# Monoculture Treated - Mechanical Automatic Modeling
Decode processed data, fit treated model zoo, and run sensitivity/uncertainty analysis.

In [2]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

  Activating project at `~/Desktop/Research/CancerGrowthDynamics/Modeling_Approaches/02_mechanical_automatic_package`


In [4]:
using CSV, DataFrames, Plots, Dates
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

In [6]:
condition = "monoculture_treated"
decoded = MechanicalAutomaticModeling.IOUtils.decode_condition_dataframe(condition; start=@__DIR__)
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
decoded_path = joinpath(out.csv, "$(condition)_automatic_decoded.csv")
CSV.write(decoded_path, decoded)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="decode", outputs=[decoded_path], start=@__DIR__)
first(decoded, min(10, nrow(decoded)))

Row,time,count,source_file,condition,density,cell_line,dose,mix
,Float64,Float64,String,String,String,String,Float64,String
1,1.0,470.381,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
2,1.0,164.837,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
3,1.0,305.19,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
4,1.0,683.095,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
5,1.0,277.199,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
6,1.0,507.502,A2780Naive_day_averages.csv,monoculture_treated,,,0.0,
7,1.0,332.641,A2780Naive_sample_averages.csv,monoculture_treated,,,0.0,
8,1.0,497.382,A2780Naive_sample_averages.csv,monoculture_treated,,,0.0,
9,1.0,581.122,A2780Naive_sample_averages.csv,monoculture_treated,,,0.0,


In [7]:
fit_artifacts = MechanicalAutomaticModeling.FitWorkflows.run_condition_fit!(decoded, condition; start=@__DIR__)
first(fit_artifacts.ranking, min(10, nrow(fit_artifacts.ranking)))

Row,model,sse,weighted_sse,aic,bic,n_params,delta_bic
,String,Float64,Float64,Float64,Float64,Int64,Float64
1,theta_logistic_hill_kill,1.0e12,1.0e12,14205.2,14232.2,6,0.0
2,transit_chain_erlang,1.0e12,1.0e12,14205.2,14232.2,6,0.0
3,adaptive_ic50,1.0e12,1.0e12,14205.2,14232.2,6,0.0
4,pkpd_inhibition,1.0e12,1.0e12,14207.2,14238.7,7,6.51026
5,sensitive_resistant,1.0e12,1.0e12,14207.2,14238.7,7,6.51026


In [8]:
analysis_artifacts = MechanicalAutomaticModeling.AnalysisWorkflows.run_condition_analysis!(decoded, fit_artifacts, condition; start=@__DIR__)
analysis_artifacts.sensitivity

Row,model,note
,String,String
1,theta_logistic_hill_kill,Sensitivity call not available with current signature; update wrapper.


In [9]:
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
summary = DataFrame(
    condition = [condition],
    decoded_rows = [nrow(decoded)],
    fit_rows = [nrow(fit_artifacts.ranking)],
    sensitivity_rows = [nrow(analysis_artifacts.sensitivity)],
    generated_at = [Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SS")]
)
summary_path = joinpath(out.metrics, "$(condition)_automatic_summary.csv")
CSV.write(summary_path, summary)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="summary", outputs=[summary_path], start=@__DIR__)
summary

Row,condition,decoded_rows,fit_rows,sensitivity_rows,generated_at
,String,Int64,Int64,Int64,String
1,monoculture_treated,672,5,1,2026-04-23T23:21:38
